In [ ]:
# @title Setup Environment
import os

!sudo apt-get update && sudo apt-get install -y ffmpeg

if not os.path.exists("heartlib"):
    !git clone https://github.com/HeartMuLa/heartlib.git

%cd heartlib

!pip install -e .
!pip install -u "huggingface_hub[cli]"
!pip install flash-attn --no-build-isolation
!pip install accelerate

In [ ]:
# @title Download Checkpoints
import os

if os.path.basename(os.getcwd()) != "heartlib":
    %cd heartlib

os.makedirs('./ckpt', exist_ok=True)

!huggingface-cli download --local-dir './ckpt' 'HeartMuLa/HeartMuLaGen'
!huggingface-cli download --local-dir './ckpt/HeartMuLa-oss-3B' 'HeartMuLa/HeartMuLa-oss-3B'
!huggingface-cli download --local-dir './ckpt/HeartCodec-oss' 'HeartMuLa/HeartCodec-oss'
!huggingface-cli download --local-dir './ckpt/HeartTranscriptor-oss' 'HeartMuLa/HeartTranscriptor-oss'

In [ ]:
# @title Run Music Generation
# No Touchy
import os
import re
import time
import shutil
import tempfile
import subprocess
from pathlib import Path
from IPython.display import Audio, display
import torch

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'

OUTPUT_DIR = Path("./output")

def get_mp3_duration(filepath: str) -> int:
    try:
        # Use ffprobe (installed via ffmpeg) to avoid extra python deps
        result = subprocess.run(
            ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", filepath],
            check=True,
            capture_output=True,
            text=True,
        )
        return int(float(result.stdout.strip()))
    except Exception:
        return 0

def sanitize_filename(lyrics: str) -> str:
    # Remove structure tags like [Intro], [Verse], [Chorus], etc.
    cleaned = re.sub(r"\[[\w\s]+\]", "", lyrics)
    text = cleaned[:20]
    text = text.replace(" ", "_")
    text = re.sub(r"[^a-zA-Z0-9_]", "", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text
    
def save_to_output(temp_path: str, lyrics: str) -> str:
    OUTPUT_DIR.mkdir(exist_ok=True)
    duration = get_mp3_duration(temp_path)
    lyrics_part = sanitize_filename(lyrics)
    timestamp = int(time.time())
    filename = f"music-{duration}-{lyrics_part}-{timestamp}.mp3"
    output_path = OUTPUT_DIR / filename
    shutil.copy2(temp_path, output_path)
    return str(output_path)

# -----------------------------------------------------------------------------
# 1. SETUP YOUR LYRICS AND TAGS (you can touch this lol)
# -----------------------------------------------------------------------------
my_lyrics = """
[Verse]
The sun creeps in across the floor
I hear the traffic outside the door
The coffee pot begins to hiss
It is another morning just like this

[Chorus]
Every day the light returns
Every day the fire burns
"""

my_tags = "piano,happy,pop"
# -----------------------------------------------------------------------------
# Do Not Touch This Code Below, lol.

if os.path.basename(os.getcwd()) != "heartlib":
    if os.path.exists("heartlib"):
        %cd heartlib
    else:
        raise FileNotFoundError("Repo not found. Please run Block 1 (Setup) first.")

# Write lyrics/tags to temp files to avoid shell quoting issues
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, encoding="utf-8") as lyrics_file:
    lyrics_file.write(my_lyrics)
    lyrics_path = lyrics_file.name

with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, encoding="utf-8") as tags_file:
    tags_file.write(my_tags)
    tags_path = tags_file.name

output_path = "./assets/output.mp3"

!python ./examples/run_music_generation.py \
    --model_path=./ckpt \
    --version="3B" \
    --lyrics="{lyrics_path}" \
    --tags="{tags_path}" \
    --save_path="{output_path}" \
    --lazy_load=true

os.unlink(lyrics_path)
os.unlink(tags_path)

if os.path.exists(output_path):
    saved_path = save_to_output(output_path, my_lyrics)
    print(f"✓ Generation complete! Saved to: {saved_path}")
    display(Audio(saved_path))
else:
    print("✗ Generation failed. Check logs above.")
